# Paper 4 — 01 · Build + freeze contrastive sets

Build the five contrastive cells (EXPERIMENT_DESIGN §2): `harm_en`, `benign_en`, `harm_ro`, `benign_ro`, and the EN<->RO `parallel` set. Read sources from Paper 2 (RoSafetyBench) and HarmBench; label behavior with the Paper 2 `gpt-5-mini` judge for the execution probe. Freeze + SHA-256 the sets and the probe split — pre-registration (EXPERIMENT_DESIGN §11).

**Output:** `data/contrastive/<short>/*.jsonl`, `data/splits/probe_split.json`, both SHA-256'd.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


## Sources

- `harm_en`: HarmBench standard + local core from crosslingual `text_en`.
- `benign_en`: XSTest-safe (matches RO over-refusal benign-but-risky semantics).
- `harm_ro` / `benign_ro`: RoSafetyBench (`paper2-benchmark/benchmark/expanded/`).
- `parallel`: RoSafetyBench crosslingual harmful pairs (`category=='harmful'`, patching-only).

In [ ]:
import yaml
cfg = yaml.safe_load((DRIVE_ROOT / 'configs' / 'experiments.yaml').read_text())
cells_cfg = cfg['contrastive_sets']['cells']
cells_cfg

## Build all cells (EN cells need HarmBench + XSTest from HF)

On Colab `datasets` + network are available, so pass `with_en=True` to
build the EN cells too. The RO cells + parallel + harm_en core are
deterministic from the committed Paper 2 files (already frozen locally;
SHA-256 in `PREREGISTRATION.md`).

In [ ]:
from contrastive import build_all
expanded = PAPER2_ROOT / 'benchmark' / 'expanded'
out = CONTRAST_DIR / short
manifest = build_all(expanded, out, with_en=True)  # with_en builds HarmBench+XSTest
import pprint; pprint.pprint(manifest)

## Behavioral labels (execution-probe target) — Paper 2 judge

In [ ]:
from llm_judge import LLMJudge   # Paper 2 src/
from judges import REFUSAL_JUDGE  # Paper 2 src/
# Greedy-generate one completion per prompt (all cells), judge refuse/comply.
# Record finish_reason + usage.{completion,reasoning}_tokens (R10 lesson).
# Write labels to data/contrastive/<short>/behavioral_labels.jsonl.


## Freeze probe split + pre-register (append to PREREGISTRATION.md)

In [ ]:
import json
# 70/30 EN train/eval split (stable, seed 17) over harm_en+benign_en;
# RO eval = harm_ro+benign_ro; parallel is patching-only.
# Write data/splits/probe_split.json + record its SHA-256.
# Append the finalized harm_en/benign_en + split SHAs to PREREGISTRATION.md §3.
